# 워크플로우 데이터 흐름 분석

이 노트북은 LangGraph 워크플로우에서 데이터가 어떻게 흘러가는지 추적하고 시각화합니다.

## 워크플로우 구조
```
START → load → crop → enhance → [filter, metrics] (병렬) → packaging → END
```

## 데이터 흐름
1. **StateGraph**: 모든 노드가 공유하는 상태 객체 (`GraphState`)
2. **Partial State**: 각 노드는 업데이트할 필드만 반환
3. **최종 저장**: `main.py`에서 `result` 상태를 읽어 파일로 저장


In [ ]:
import sys
sys.path.append('..')

from src.state import GraphState
from src.graph_builder import build_graph
from src.nodes.load import load_node
from src.nodes.crop import crop_node
from src.nodes.enhancement import enhancement_node
from src.nodes.filter import filter_node
from src.nodes.metrics import metrics_node
from src.nodes.packaging import packaging_node
import numpy as np
from pathlib import Path


## 1. GraphState 구조 확인


In [ ]:
# GraphState 필드 확인
print("=" * 60)
print("GraphState 구조")
print("=" * 60)
print("\n입력:")
print("  - input_image_path: str")

print("\n처리 단계별 이미지:")
print("  - original_image: Optional[np.ndarray]")
print("  - cropped_image: Optional[np.ndarray]")
print("  - enhanced_image: Optional[np.ndarray]")
print("  - filtered_image: Optional[np.ndarray]")
print("  - binary_mask: Optional[np.ndarray]")

print("\n분석 결과:")
print("  - metrics: Optional[dict]")
print("  - analysis_data: Optional[dict]")

print("\n에러 수집:")
print("  - errors: Annotated[list[str], operator.add]")

print("\n" + "=" * 60)


## 2. 각 노드의 데이터 흐름 추적


In [ ]:
# 각 노드가 반환하는 Partial State 확인
print("=" * 60)
print("각 노드의 Partial State 반환값")
print("=" * 60)

print("\n1. load_node:")
print("   입력: state['input_image_path']")
print("   반환: {'original_image': np.ndarray}")
print("   설명: 이미지 파일을 로드하여 original_image에 저장")

print("\n2. crop_node:")
print("   입력: state['original_image']")
print("   반환: {'cropped_image': np.ndarray}")
print("   설명: 원본 이미지를 크롭하여 cropped_image에 저장")

print("\n3. enhancement_node:")
print("   입력: state['cropped_image']")
print("   반환: {'enhanced_image': np.ndarray}")
print("   설명: Real-ESRGAN으로 4x 확대하여 enhanced_image에 저장")

print("\n4. filter_node (병렬):")
print("   입력: state['enhanced_image']")
print("   반환: {'filtered_image': np.ndarray}")
print("   설명: CLAHE 필터 적용하여 filtered_image에 저장")

print("\n5. metrics_node (병렬):")
print("   입력: state['enhanced_image']")
print("   반환: {'binary_mask': np.ndarray, 'metrics': dict}")
print("   설명: 형태학적 분석하여 binary_mask와 metrics에 저장")

print("\n6. packaging_node:")
print("   입력: state['enhanced_image'], state['filtered_image'],")
print("         state['metrics'], state['binary_mask']")
print("   반환: {'analysis_data': dict}")
print("   설명: 모든 데이터를 JSON 형식으로 패키징하여 analysis_data에 저장")

print("\n" + "=" * 60)


## 3. 실제 워크플로우 실행 및 상태 추적


In [ ]:
# 실제 워크플로우 실행 (간단한 예시)
# 실제 이미지 파일이 필요하므로, 상태 추적만 시뮬레이션

print("=" * 60)
print("워크플로우 실행 시뮬레이션")
print("=" * 60)

# 초기 상태
initial_state = {
    "input_image_path": "data/example.png",
    "errors": []
}

print("\n[초기 상태]")
print(f"  input_image_path: {initial_state['input_image_path']}")
print(f"  original_image: {initial_state.get('original_image', None)}")
print(f"  errors: {initial_state['errors']}")

# 각 단계별 상태 변화 시뮬레이션
print("\n[단계별 상태 변화]")

print("\n1. load_node 실행 후:")
state_after_load = {**initial_state, "original_image": "<np.ndarray>"}
print(f"   ✓ original_image: 추가됨")
print(f"   상태 키: {list(state_after_load.keys())}")

print("\n2. crop_node 실행 후:")
state_after_crop = {**state_after_load, "cropped_image": "<np.ndarray>"}
print(f"   ✓ cropped_image: 추가됨")
print(f"   상태 키: {list(state_after_crop.keys())}")

print("\n3. enhancement_node 실행 후:")
state_after_enhance = {**state_after_crop, "enhanced_image": "<np.ndarray>"}
print(f"   ✓ enhanced_image: 추가됨")
print(f"   상태 키: {list(state_after_enhance.keys())}")

print("\n4. filter_node 실행 후 (병렬):")
state_after_filter = {**state_after_enhance, "filtered_image": "<np.ndarray>"}
print(f"   ✓ filtered_image: 추가됨")
print(f"   상태 키: {list(state_after_filter.keys())}")

print("\n5. metrics_node 실행 후 (병렬):")
state_after_metrics = {
    **state_after_filter, 
    "binary_mask": "<np.ndarray>",
    "metrics": {"circularity": 0.115, "solidity": 0.706, "area": 339401}
}
print(f"   ✓ binary_mask: 추가됨")
print(f"   ✓ metrics: 추가됨")
print(f"   상태 키: {list(state_after_metrics.keys())}")

print("\n6. packaging_node 실행 후:")
state_after_packaging = {
    **state_after_metrics,
    "analysis_data": {
        "metadata": {...},
        "images": {...},
        "metrics": {...},
        "comparison": {...}
    }
}
print(f"   ✓ analysis_data: 추가됨")
print(f"   상태 키: {list(state_after_packaging.keys())}")

print("\n" + "=" * 60)


## 4. main.py에서의 데이터 저장 과정


In [ ]:
# main.py의 저장 과정 시뮬레이션
print("=" * 60)
print("main.py에서의 데이터 저장 과정")
print("=" * 60)

print("\n1. 그래프 실행:")
print("   result = graph.invoke(initial_state)")
print("   → result는 최종 GraphState 객체")

print("\n2. result에서 데이터 추출:")
print("   analysis_data = result['analysis_data']")
print("   → packaging_node에서 생성된 JSON 데이터")

print("\n3. Format 2 변환:")
print("   gemini_format = to_gemini_vertex_ai_format(analysis_data)")
print("   → analysis_data를 Format 2 형식으로 변환")
print("   → 반환값: [text, image1, image2, image3] 리스트")

print("\n4. JSON 파일 저장:")
print("   llm_gemini_format.json에 gemini_format 저장")
print("   → Part 1: 텍스트 (순수 데이터 설명)")
print("   → Part 2-4: Base64 인코딩된 이미지")

print("\n5. 이미지 시각화 저장:")
print("   full_pipeline.png 생성")
print("   → result['original_image'], result['cropped_image'],")
print("     result['enhanced_image'], result['filtered_image'],")
print("     result['binary_mask']를 사용하여 시각화")

print("\n" + "=" * 60)
print("\n중요:")
print("  - 워크플로우 내부: StateGraph가 상태를 관리")
print("  - 각 노드: Partial State만 반환 (업데이트할 필드만)")
print("  - 최종 저장: main.py에서 result 상태를 읽어 파일로 저장")
print("  - 저장 위치: outputs/{input_filename}/ 디렉토리")
print("=" * 60)


## 5. 데이터 흐름 다이어그램


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# 데이터 흐름 다이어그램 생성
fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')

# 색상 정의
colors = {
    'node': '#4A90E2',
    'state': '#50C878',
    'save': '#FF6B6B',
    'arrow': '#333333'
}

# 노드 위치 정의
nodes = {
    'START': (1, 10),
    'load': (3, 10),
    'crop': (5, 10),
    'enhance': (7, 10),
    'filter': (6, 8),
    'metrics': (8, 8),
    'packaging': (7, 6),
    'END': (7, 4),
    'main_save': (9, 6)
}

# 상태 박스 위치
state_boxes = {
    'original_image': (3, 8.5),
    'cropped_image': (5, 8.5),
    'enhanced_image': (7, 8.5),
    'filtered_image': (6, 6.5),
    'binary_mask': (8, 6.5),
    'metrics': (8, 5.5),
    'analysis_data': (7, 4.5)
}

# 노드 그리기
for name, pos in nodes.items():
    if name in ['START', 'END']:
        ax.add_patch(FancyBboxPatch(
            (pos[0]-0.4, pos[1]-0.3), 0.8, 0.6,
            boxstyle="round,pad=0.1",
            facecolor=colors['node'],
            edgecolor='black',
            linewidth=2
        ))
        ax.text(pos[0], pos[1], name, ha='center', va='center', 
                fontsize=10, fontweight='bold', color='white')
    elif name == 'main_save':
        ax.add_patch(FancyBboxPatch(
            (pos[0]-0.5, pos[1]-0.3), 1.0, 0.6,
            boxstyle="round,pad=0.1",
            facecolor=colors['save'],
            edgecolor='black',
            linewidth=2
        ))
        ax.text(pos[0], pos[1], 'main.py\n저장', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
    else:
        ax.add_patch(FancyBboxPatch(
            (pos[0]-0.5, pos[1]-0.3), 1.0, 0.6,
            boxstyle="round,pad=0.1",
            facecolor=colors['node'],
            edgecolor='black',
            linewidth=2
        ))
        ax.text(pos[0], pos[1], name, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')

# 상태 박스 그리기
for name, pos in state_boxes.items():
    ax.add_patch(FancyBboxPatch(
        (pos[0]-0.6, pos[1]-0.2), 1.2, 0.4,
        boxstyle="round,pad=0.05",
        facecolor=colors['state'],
        edgecolor='black',
        linewidth=1,
        alpha=0.7
    ))
    ax.text(pos[0], pos[1], name, ha='center', va='center',
            fontsize=7, fontweight='bold')

# 화살표 그리기
arrows = [
    # 순차 흐름
    ((1.4, 10), (2.5, 10), ''),
    ((3.5, 10), (4.5, 10), ''),
    ((5.5, 10), (6.5, 10), ''),
    # 병렬 흐름
    ((7, 9.7), (6, 8.6), ''),
    ((7, 9.7), (8, 8.6), ''),
    # 수집
    ((6, 7.7), (6.5, 6.6), ''),
    ((8, 7.7), (7.5, 6.6), ''),
    # 종료
    ((7, 5.7), (7, 4.6), ''),
    # 저장
    ((7, 3.7), (8.5, 6), ''),
]

for start, end, label in arrows:
    arrow = FancyArrowPatch(start, end,
                           arrowstyle='->', lw=2,
                           color=colors['arrow'],
                           connectionstyle='arc3,rad=0.1')
    ax.add_patch(arrow)

# 제목 및 범례
ax.text(5, 11.5, 'LangGraph 워크플로우 데이터 흐름', 
        ha='center', fontsize=14, fontweight='bold')

# 범례
legend_elements = [
    mpatches.Patch(facecolor=colors['node'], label='노드'),
    mpatches.Patch(facecolor=colors['state'], label='상태 필드', alpha=0.7),
    mpatches.Patch(facecolor=colors['save'], label='저장 작업')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

print("\n✓ 데이터 흐름 다이어그램 생성 완료")


In [ ]:
# 실제 실행 예시 (주석 처리됨)
# 실제 이미지 파일이 필요합니다

"""
from pathlib import Path
import sys
sys.path.append('..')

from src.graph_builder import build_graph
from src.utils import find_data_directory

# 데이터 디렉토리 찾기
try:
    data_dir = find_data_directory()
    image_files = list(Path(data_dir).glob("*.png"))
    
    if image_files:
        # 첫 번째 이미지 선택
        test_image = str(image_files[0])
        print(f"테스트 이미지: {test_image}")
        
        # 그래프 빌드
        graph = build_graph()
        
        # 초기 상태
        initial_state = {
            "input_image_path": test_image,
            "errors": []
        }
        
        # 그래프 실행
        result = graph.invoke(initial_state)
        
        # 상태 확인
        print("\n최종 상태 키:")
        for key in result.keys():
            if key != 'errors':
                value = result[key]
                if isinstance(value, np.ndarray):
                    print(f"  {key}: np.ndarray {value.shape}")
                elif isinstance(value, dict):
                    print(f"  {key}: dict (keys: {list(value.keys())})")
                else:
                    print(f"  {key}: {type(value).__name__}")
        
        # analysis_data 확인
        if result.get('analysis_data'):
            print("\nanalysis_data 구조:")
            print(f"  metadata: {list(result['analysis_data']['metadata'].keys())}")
            print(f"  images: {list(result['analysis_data']['images'].keys())}")
            print(f"  metrics: {list(result['analysis_data']['metrics'].keys())}")
            print(f"  comparison: {list(result['analysis_data']['comparison'].keys())}")
        
        print("\n✓ 워크플로우 실행 완료")
    else:
        print("이미지 파일을 찾을 수 없습니다.")
        
except Exception as e:
    print(f"오류 발생: {e}")
"""
